In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from wordcloud import WordCloud
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon', quiet=True)
from bertopic import BERTopic
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Replace with your actual file path
df = pd.read_csv('cleaned_reviews.csv')
print(df.shape)
df.head()

In [ ]:
import pandas as pd

# Define the subsets you want to evaluate
subsets_to_check = {
    'right now' : ["ProductId", "UserId", "Time", "Text"],
    'User + Product': ['UserId', 'ProductId'],
    'User + Product + Cleaned Text': ['UserId', 'ProductId', 'cleaned_text'],
    'Cleaned Text Only': ['cleaned_text'],
    'User + Cleaned Text': ['UserId', 'cleaned_text']
}

# Calculate stats for each subset
stats = []
for name, cols in subsets_to_check.items():
    total_rows = len(df)
    unique_combos = df.drop_duplicates(subset=cols).shape[0]
    rows_to_remove = total_rows - unique_combos
    
    # Frequency analysis
    combo_counts = df[cols].value_counts()
    dup_groups = (combo_counts > 1).sum()  # How many unique combos appear >1 time
    max_copies = combo_counts.max()        # Highest duplication count for a single combo
    
    stats.append({
        'Subset': name,
        'Total Rows': total_rows,
        'Rows to Remove': rows_to_remove,
        '% Data Affected': f"{(rows_to_remove / total_rows) * 100:.2f}%",
        'Duplicate Groups': dup_groups,
        'Max Copies of 1 Group': max_copies
    })

# Display as a clean table
stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

In [ ]:
# Pick a subset to inspect, e.g., 'Cleaned Text Only'
subset = ['cleaned_text', 'UserId']
top_dups = df[subset].value_counts().head(10)

print("Top 10 Most Duplicated Combinations:")
for combo, count in top_dups.items():
    print(f"\n{'-'*40}\nCount: {count}\n{combo}")

In [ ]:
# 🌟 Updated WordCloud with Stopwords Filtering
from wordcloud import STOPWORDS
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

# 1️⃣ Combine built-in WordCloud stopwords + NLTK English stopwords
stop_words = STOPWORDS.union(set(nltk_stopwords.words('english')))

# 2️⃣ Add custom filler words common in Amazon reviews
# ⚠️ FIX: Original had missing commas — Python silently joined adjacent strings
# (e.g., 'though' 'use' became 'thoughuse'). All commas now present.
stop_words.update([
    # HTML artifacts
    'br', 'href', 'http', 'www', 'html',
    
    # Generic verbs (not informative)
    'dont', 'cant', 'wont', 'didnt', 'doesnt', 'isnt', 'wasnt', 'arent', 'without',
    'couldnt', 'shouldnt', 'wouldnt',
    'say', 'said', 'tell', 'told', 'think', 'thought', 'know', 'knew',
    'see', 'saw', 'look', 'looked', 'want', 'wanted', 'need', 'needed',
    'get', 'got', 'gotten', 'make', 'made', 'take', 'took', 'give', 'gave',
    'go', 'went', 'gone', 'come', 'came', 'find', 'found', 'try', 'tried', 'though',
    'use', 'used', 'buy', 'bought',
    
    # Vague quantifiers/adverbs
    'much', 'many', 'little', 'lot', 'lots', 'way', 'still', 'well',
    'first', 'last', 'next', 'new', 'old', 'good', 'great', 'best',
    'better', 'bad', 'worst', 'nice', 'fine', 'pretty', 'really',
    'just', 'also', 'too', 'very', 'so', 'even', 'ever', 'never',
    'definitely', 'absolutely', 'totally', 'completely', 'extremely',
    
    # Common review filler
    'product', 'products', 'item', 'items', 'thing', 'things', 'stuff',
    'review', 'reviews', 'amazon', 'seller',
    'time', 'times', 'day', 'days', 'week', 'month', 'year', 'years',
    'one', 'two', 'three', 'four', 'five',
    'im', 'ive', 'youre', 'theyre', 'weve', 'theres', 'heres',
    'would', 'could', 'should', 'will', 'can', 'may', 'might', 'must',
    'than', 'then', 'now', 'here', 'there', 'where', 'when', 'why', 'how', 'add'
])

# 3️⃣ Generate WordCloud
all_text = ' '.join(df['cleaned_text'].str.lower().sample(frac=0.5, random_state=42))
wc = WordCloud(
    width=800, 
    height=400, 
    background_color='white', 
    max_words=150,
    stopwords=stop_words,
    collocations=True
).generate(all_text)

plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Reviews (Stopwords Removed)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1️⃣ Basic statistics
print("📈 HelpfulnessNumerator - Basic Stats:")
print(df['HelpfulnessNumerator'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

# 2️⃣ Count of reviews with zero helpfulness votes
zero_votes = (df['HelpfulnessNumerator'] == 0).sum()
print(f"\n🔹 Reviews with 0 helpful votes: {zero_votes:,} ({zero_votes/len(df)*100:.1f}%)")

# 5️⃣ CDF
plt.figure(figsize=(8, 5))
sorted_votes = np.sort(df['HelpfulnessNumerator'])
cdf = np.arange(1, len(sorted_votes) + 1) / len(sorted_votes)
plt.plot(sorted_votes, cdf, linewidth=2, color='navy')
plt.axvline(x=10, color='red', linestyle='--', label='Threshold = 10')
plt.axvline(x=20, color='green', linestyle='--', label='Threshold = 20')
plt.axvline(x=50, color='orange', linestyle='--', label='Threshold = 50')
plt.title('Cumulative Distribution of HelpfulnessNumerator')
plt.xlabel('HelpfulnessNumerator (Threshold)')
plt.ylabel('Cumulative % of Reviews')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(0, 100)
plt.show()

# 6️⃣ Quick reference table
print("\n📋 Reviews retained at different thresholds:")
thresholds = [0, 2, 5, 10, 15, 20, 30, 50, 100]
for t in thresholds:
    count = (df['HelpfulnessNumerator'] >= t).sum()
    pct = count / len(df) * 100
    print(f"  ≥ {t:3d} votes → {count:7,} reviews ({pct:5.2f}%)")

In [ ]:
HELPFUL_THRESHOLD = 2

df_model = df[df['HelpfulnessNumerator'] >= HELPFUL_THRESHOLD].copy().reset_index(drop=True)

print(f"📉 Dataset size after filtering: {len(df_model):,} reviews ({len(df_model)/len(df)*100:.1f}% of original)")
print(f"📊 Score distribution (check for bias):")
print(df_model['Score'].value_counts(normalize=True).sort_index().round(3))

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words=list(stop_words))

# Optimized for speed & memory on filtered subset
topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2", 
    language="english",
    vectorizer_model=vectorizer_model,   
    min_topic_size=500,                  
    calculate_probabilities=False,
    verbose=False
)

topics, probs = topic_model.fit_transform(df_model['cleaned_text'].tolist())
df_model['topic'] = topics

# Get topic info
topic_info = topic_model.get_topic_info()
print(topic_info)

In [ ]:
topic_model.visualize_topics(top_n_topics=15)

In [ ]:
topic_model.visualize_barchart(top_n_topics=15)

## 💬 Sentiment Analysis per Topic

Now that we have topic clusters, let's measure customer sentiment within each topic.
This tells us **what people feel** about each product segment — not just what they're talking about.

**Why this matters:** Two topics with identical avg ratings can have very different sentiment patterns.
A topic full of polarized 1s and 5s reads differently from a topic of lukewarm 3s and 4s,
and the *language* in negative reviews shows you exactly what's failing in that segment.

In [ ]:
# 1️⃣ Run VADER sentiment on each review
# ⚠️ FIX: Original code referenced 'clean_text' but the column is 'cleaned_text'
sia = SentimentIntensityAnalyzer()

# Run on cleaned_text (sentiment performs OK on cleaned text; VADER handles it well)
df_model['sentiment_scores'] = df_model['cleaned_text'].apply(sia.polarity_scores)

# Extract individual components for richer analysis
df_model['sent_compound'] = df_model['sentiment_scores'].apply(lambda x: x['compound'])
df_model['sent_pos'] = df_model['sentiment_scores'].apply(lambda x: x['pos'])
df_model['sent_neu'] = df_model['sentiment_scores'].apply(lambda x: x['neu'])
df_model['sent_neg'] = df_model['sentiment_scores'].apply(lambda x: x['neg'])

# Categorize using standard VADER thresholds
def label_sentiment(c):
    if c >= 0.05:
        return 'positive'
    elif c <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df_model['sentiment'] = df_model['sent_compound'].apply(label_sentiment)

# Quick sanity check
print("📊 Overall sentiment distribution:")
print(df_model['sentiment'].value_counts(normalize=True).round(3))
print(f"\n📊 Avg compound score: {df_model['sent_compound'].mean():.3f}")
print(f"📊 Median compound score: {df_model['sent_compound'].median():.3f}")

In [ ]:
# 2️⃣ Sanity check: does VADER sentiment align with star ratings?
# If yes, our sentiment signal is trustworthy.
sentiment_vs_rating = df_model.groupby('Score')['sent_compound'].agg(['mean', 'median', 'count']).round(3)
print("📊 Sentiment compound score by Star Rating (should increase monotonically):")
print(sentiment_vs_rating)

# Cross-tab: how often sentiment label matches rating expectation?
print("\n📊 Sentiment label distribution by Score:")
crosstab = pd.crosstab(df_model['Score'], df_model['sentiment'], normalize='index').round(3)
print(crosstab)

In [ ]:
# 3️⃣ Aggregate sentiment per topic (excluding outlier topic -1)
df_analysis = df_model[df_model['topic'] != -1].copy()

# Combine sentiment proportions + average compound score per topic
sentiment_props = (
    df_analysis.groupby('topic')['sentiment']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)
)

sentiment_stats = df_analysis.groupby('topic').agg(
    avg_compound=('sent_compound', 'mean'),
    median_compound=('sent_compound', 'median'),
).round(3)

# Merge with topic info
topic_sentiment = (
    topic_info[topic_info['Topic'] != -1][['Topic', 'Count', 'Name']]
    .merge(sentiment_props, left_on='Topic', right_index=True, how='left')
    .merge(sentiment_stats, left_on='Topic', right_index=True, how='left')
    .round(3)
)

print("📊 Sentiment distribution per topic (top 15 by review count):")
print(topic_sentiment.sort_values('Count', ascending=False).head(15).to_string(index=False))

In [ ]:
# 4️⃣ Visualize: stacked sentiment bars per topic
plot_data = topic_sentiment.sort_values('Count', ascending=False).head(15).copy()
plot_data['short_name'] = plot_data['Name'].apply(lambda x: x[:35] + '...' if len(str(x)) > 35 else x)

fig, ax = plt.subplots(figsize=(12, 7))
y_pos = np.arange(len(plot_data))

ax.barh(y_pos, plot_data['negative'], color='#d62728', label='Negative', edgecolor='white')
ax.barh(y_pos, plot_data['neutral'], left=plot_data['negative'],
        color='#bdbdbd', label='Neutral', edgecolor='white')
ax.barh(y_pos, plot_data['positive'],
        left=plot_data['negative'] + plot_data['neutral'],
        color='#2ca02c', label='Positive', edgecolor='white')

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_data['short_name'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Sentiment Proportion')
ax.set_xlim(0, 1)
ax.set_title('Sentiment Composition per Topic (Top 15 by Volume)', pad=15)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 5️⃣ Combined business view: sentiment + ratings + product diversity per topic
topic_metrics = df_analysis.groupby('topic').agg(
    review_count=('Score', 'count'),
    avg_rating=('Score', 'mean'),
    median_rating=('Score', 'median'),
    std_rating=('Score', 'std'),
    unique_products=('ProductId', 'nunique'),
    unique_users=('UserId', 'nunique'),
    avg_compound=('sent_compound', 'mean'),
    pct_negative=('sentiment', lambda s: (s == 'negative').mean()),
    pct_positive=('sentiment', lambda s: (s == 'positive').mean()),
).reset_index()

topic_metrics = topic_metrics.merge(
    topic_info[['Topic', 'Name']], left_on='topic', right_on='Topic', how='left'
)

topic_metrics['top_keywords'] = topic_metrics['topic'].apply(
    lambda t: ', '.join([word for word, _ in topic_model.get_topic(t)][:4])
)

cols = ['Name', 'top_keywords', 'review_count', 'unique_products',
        'avg_rating', 'avg_compound', 'pct_negative', 'pct_positive']

print("📈 Topic Metrics with Sentiment (Top 15 by review count):")
print(
    topic_metrics.sort_values('review_count', ascending=False)
    .head(15)[cols].round(3).to_string(index=False)
)

In [ ]:
# 6️⃣ Original chart: Avg Rating + Unique Products per Topic
plot_data = topic_metrics.sort_values('avg_rating', ascending=True).head(15)

fig, ax1 = plt.subplots(figsize=(14, 7))

# Avg Rating bars
bars = ax1.barh(range(len(plot_data)), plot_data['avg_rating'],
                color=plt.cm.RdYlGn((plot_data['avg_rating'] - 1) / 4),
                edgecolor='black', alpha=0.85, height=0.6, label='Avg Rating')

ax1.set_yticks(range(len(plot_data)))
ax1.set_yticklabels([f"{row['Name'][:30]}..." if len(str(row['Name'])) > 30 else row['Name']
                     for _, row in plot_data.iterrows()], fontsize=9)
ax1.set_xlabel('Average Rating (1–5)', color='black')
ax1.set_xlim(1, 5)

# Unique product dots
ax2 = ax1.twiny()
ax2.scatter(plot_data['unique_products'], range(len(plot_data)),
            color='steelblue', zorder=5, s=80, marker='D', label='Unique Products')

for i, val in enumerate(plot_data['unique_products']):
    ax2.plot([0, val], [i, i], color='steelblue', linewidth=0.8, alpha=0.4, linestyle=':')

ax2.set_xlabel('Unique Product Count', color='steelblue')
ax2.tick_params(axis='x', labelcolor='steelblue')
ax2.set_xlim(0, plot_data['unique_products'].max() * 1.2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=9)

plt.title('Average Rating & Unique Product Count per Topic (Top 15)', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# 7️⃣ Opportunity Quadrant: Sentiment vs Product Saturation
# This is the key business chart — combines sentiment with market saturation.
#  - Bottom-right (positive sentiment, few products) → ENTER: validated demand, low competition
#  - Bottom-left  (negative sentiment, few products) → IMPROVE: customer pain, room to fix
#  - Top-right    (positive sentiment, many products) → CROWDED: avoid unless differentiated
#  - Top-left     (negative sentiment, many products) → DISRUPT: many bad incumbents to beat

plot_df = topic_metrics[topic_metrics['review_count'] >= 100].copy()

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    plot_df['avg_compound'],
    plot_df['unique_products'],
    s=plot_df['review_count'] / 5,
    c=plot_df['avg_rating'],
    cmap='RdYlGn',
    vmin=1, vmax=5,
    alpha=0.7,
    edgecolors='black',
    linewidth=0.8,
)

# Median lines for quadrant split
ax.axvline(plot_df['avg_compound'].median(), color='gray', linestyle='--', alpha=0.5)
ax.axhline(plot_df['unique_products'].median(), color='gray', linestyle='--', alpha=0.5)

# Annotate every point
for _, row in plot_df.iterrows():
    label = str(row['Name'])[:25]
    ax.annotate(label, (row['avg_compound'], row['unique_products']),
                fontsize=7, alpha=0.85,
                xytext=(5, 5), textcoords='offset points')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Avg Star Rating')

ax.set_xlabel('Avg Sentiment Compound Score (negative ← → positive)')
ax.set_ylabel('Unique Products in Topic (market saturation)')
ax.set_title('Opportunity Quadrant: Sentiment vs Market Saturation\n(bubble size = review volume)', pad=15)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 8️⃣ Drill-down: most negative reviews per topic — the WHY behind low sentiment
# Useful for understanding specific pain points to fix or differentiate against.

def show_top_negative(topic_id, n=3):
    topic_name = topic_info.loc[topic_info['Topic'] == topic_id, 'Name'].values[0]
    subset = df_analysis[df_analysis['topic'] == topic_id].nsmallest(n, 'sent_compound')
    print(f"\n{'='*70}\nTopic {topic_id}: {topic_name}\n{'='*70}")
    for _, row in subset.iterrows():
        text = row['cleaned_text'][:300]
        print(f"  ★{row['Score']} | compound={row['sent_compound']:.3f}")
        print(f"    \"{text}...\"\n")

# Show negatives for the 5 topics with the worst average sentiment
worst_topics = topic_metrics.sort_values('avg_compound').head(5)['topic'].tolist()
print("🔻 Top negative reviews from the 5 lowest-sentiment topics:")
for t in worst_topics:
    show_top_negative(t, n=2)

### Recommendations (Updated with Sentiment)

1. **Topic segmentation** identifies clear product categories: kitchen spices and salts (8), pasta noodle products (2), dog foods and products (6), etc.

2. **Three signals together — rating, sentiment, and product diversity — give a richer market read** than rating alone:
   - **Avg star rating** = explicit user judgment (1–5 scale)
   - **VADER compound score** = implicit sentiment from review *language* (catches things ratings hide, like a 4-star review with frustrated wording)
   - **Unique product count** = market saturation

3. **Use the opportunity quadrant to prioritize:**
   - **Positive sentiment + low product count** → strong entry signal: validated demand without crowded competition
   - **Negative sentiment + low product count** → improvement play: identifiable pain points and few incumbents to displace
   - **Positive sentiment + high product count** → saturated; only enter with a clear differentiator
   - **Negative sentiment + high product count** → disruption opportunity if you can solve what existing brands fail at

4. **Watch for sentiment-rating divergence.** A topic with a 4.2 avg rating but only 60% positive sentiment likely has politely negative language users didn't reflect in stars — these are early warning signs worth investigating with the drill-down cell.

5. **The negative review drill-down (cell 8) surfaces the actual "why."** Read these before any market-entry decision — they tell you what to build differently.